In [ ]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [ ]:
df = pd.read_csv('../data/adult.csv', index_col=0)

print('Rows before cleaning:', len(df))

df = df[~(df == '?').any(axis=1)].reset_index(drop=True)

print('Rows after cleaning (no "?"):', len(df))

inconsistent = df[((df['salary'] == '>50K') & (df['salary K$'] <= 50)) |
                  ((df['salary'] == '<=50K') & (df['salary K$'] > 50))]
print('Inconsistent salary rows:', len(inconsistent))

# Task 1
Print the count of men and women in the dataset.

In [ ]:
# Task 1
counts = df['sex'].value_counts()
print('Men:', counts.get('Male', 0))
print('Women:', counts.get('Female', 0))

# Task 2
Find the average age of men in dataset

In [ ]:
# Task 2
mean_age_men = df.loc[df['sex'] == 'Male', 'age'].mean()
print('Average age of men:', round(mean_age_men, 2))

# Task 3
Get the percentage of people from Poland (native-country)

In [ ]:
# Task 3
country_counts = df['native-country'].value_counts()
n_poland = country_counts.get('Poland', 0)
percentage = n_poland / len(df) * 100
print(f'People from Poland: {n_poland} ({percentage:.2f}%)')

# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [ ]:
# Task 4
rich = df.loc[df['salary'] == '>50K', 'age']
poor = df.loc[df['salary'] == '<=50K', 'age']

print('>50K  -> mean age: {:.2f}, std: {:.2f}'.format(rich.mean(), rich.std()))
print('<=50K -> mean age: {:.2f}, std: {:.2f}'.format(poor.mean(), poor.std()))

# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [ ]:
# Task 5
HIGHER_ED = {'Bachelors', 'Prof-school', 'Assoc-acdm', 'Assoc-voc', 'Masters', 'Doctorate'}

rich_no_higher_ed = df[(df['salary'] == '>50K') & (~df['education'].isin(HIGHER_ED))]

print('People without higher education but >50K salary:', len(rich_no_higher_ed))
print()
print('Their education breakdown:')
print(rich_no_higher_ed['education'].value_counts())

# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [ ]:
# Task 6
age_stats_by_education = df.groupby('education')['age'].describe()
print(age_stats_by_education)

# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [ ]:
# Task 7
men = df[df['sex'] == 'Male'].copy()
men['is_married'] = men['marital-status'].str.startswith('Married')
men['is_rich'] = men['salary'] == '>50K'

share = men.groupby('is_married')['is_rich'].mean()
print('Share of men earning >50K:')
print(share)
print()
loser = 'Married' if share[True] > share[False] else 'Non-married'
print(f'-> {loser} men earn more (higher >>50K<< share).')

# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [ ]:
# Task 8
max_hours = df['hours-per-week'].max()
n_people = (df['hours-per-week'] == max_hours).sum()
print(f'Max hours per week: {max_hours}')
print(f'People working that many hours: {n_people}')

# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [ ]:
# Task 9
numeric = df[['age', 'hours-per-week', 'salary K$']].copy()

education_order = ['Preschool', '1st-4th', '5th-6th', '7th-8th', '9th', '10th',
                   '11th', '12th', 'HS-grad', 'Prof-school', 'Assoc-voc',
                   'Assoc-acdm', 'Some-college', 'Bachelors', 'Masters', 'Doctorate']
numeric['education_level'] = df['education'].map({e: i for i, e in enumerate(education_order)})

corr = numeric.corr()
print('Correlation matrix:')
print(corr.round(2))
print()

pairs = corr.stack().rename('r').reset_index()
pairs = pairs[pairs['level_0'] < pairs['level_1']]
pairs = pairs.reindex(pairs['r'].abs().sort_values(ascending=False).index)

print('Strongest correlations:')
for _, row in pairs.head(4).iterrows():
    print(f"  {row['level_0']:>15} <-> {row['level_1']:<15}  r = {row['r']:.2f}")

print()
print('Notes:')
print('  No feature pairs exceed |r| = 0.3, but the strongest (and most sensible)')
print('  relationships all involve salary:')
print('  - salary K$  <->  hours-per-week:  people who work more earn a bit more.')
print('  - salary K$  <->  education_level: higher education -> slightly higher income.')
print('  - salary K$  <->  age:             older workers tend to earn more.')